In [2]:
import pandas as pd
import requests

In [3]:
url = "https://api.worldbank.org/v2/country/KEN/indicator/NY.GDP.MKTP.KD.ZG?format=json&per_page=1000"

In [4]:
response = requests.get(url)
print(response.status_code)

200


In [5]:
## Lets check if the request will work
response = requests.get(url)
print(response.status_code)   # 200 means the request was successful
data = response.json()
type(data), len(data)

200


(list, 2)

In [6]:
# The part we actually care about
records = data[1]
records[0]   #peek at the first record

{'indicator': {'id': 'NY.GDP.MKTP.KD.ZG', 'value': 'GDP growth (annual %)'},
 'country': {'id': 'KE', 'value': 'Kenya'},
 'countryiso3code': 'KEN',
 'date': '2025',
 'value': None,
 'unit': '',
 'obs_status': '',
 'decimal': 1}

In [7]:
# We want to pull out date and value for every record
rows = []     # start with an empty list this will hold my cleaned data
for r in records:  #loop through each dictionary in my records list,one at a time
    rows.append({"year": r["date"], "value": r["value"]})  #for each one build a small dict with just the 2 fields ypu want and add it to rows

In [8]:
#Turn it into a dataframe and clean it up
df = pd.DataFrame(rows)
df.head() 

,year,value
0,2025,NaN
1,2024,4.724740
2,2023,5.719992
3,2022,4.859981
4,2021,7.590489


In [9]:
df.info() #check the data types and null values


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    66 non-null     object 
 1   value   64 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.2+ KB


In [10]:
df["year"] = df["year"].astype(int) #convert year from string(object) to int

In [11]:
df = df.dropna(subset=["value"])

In [12]:
df = df.sort_values("year").reset_index(drop=True) #sort by year and reset the index

In [13]:
df.head()

,year,value
0,1961,-7.774635
1,1962,9.457359
2,1963,8.778340
3,1964,4.964467
4,1965,2.009094


Lets turn this whole block into a reusable function

** What this block does **

**The function (`fetch_indicator`)**
This defines a reusable recipe for pulling one World Bank indicator. It takes two
inputs — the indicator's code (e.g. `"NY.GDP.MKTP.KD.ZG"`) and a human-readable
label (e.g. `"GDP growth"`) — and runs through the same steps every time:
build the request URL → call the API → pull out the year/value pairs →
clean up the data types → hand back a tidy DataFrame.

**The three calls**second code
This reuses that one recipe for three different indicators, storing each
result separately: `gdp_df`, `inflation_df`, `unemployment_df`. Instead of
retyping the whole fetch-and-clean process three times, we just swap in a
different code + label pair per call.

In [14]:
import time

def fetch_indicator(indicator_code, label, max_retries=3):
    url = f"https://api.worldbank.org/v2/country/KEN/indicator/{indicator_code}?format=json&per_page=1000"

    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            data = response.json()
            break
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} failed for {label}: {e}")
            time.sleep(3)
    else:
        raise Exception(f"Could not fetch {label} after {max_retries} attempts")

    records = data[1]
    rows = []
    for r in records:
        rows.append({"year": r["date"], "value": r["value"], "indicator": label})

    df = pd.DataFrame(rows)
    df["year"] = df["year"].astype(int)
    df = df.dropna(subset=["value"])
    df = df.sort_values("year").reset_index(drop=True)
    return df

In [15]:
gdp_df = fetch_indicator("NY.GDP.MKTP.KD.ZG", "GDP growth")
inflation_df = fetch_indicator("FP.CPI.TOTL.ZG", "Inflation")
unemployment_df = fetch_indicator("SL.UEM.TOTL.ZS", "Unemployment")

In [16]:
print(gdp_df.shape, inflation_df.shape, unemployment_df.shape)
gdp_df.head()

(64, 3) (65, 3) (35, 3)


,year,value,indicator
0,1961,-7.774635,GDP growth
1,1962,9.457359,GDP growth
2,1963,8.778340,GDP growth
3,1964,4.964467,GDP growth
4,1965,2.009094,GDP growth


In [17]:
inflation_df.head()

,year,value,indicator
0,1960,1.243781,Inflation
1,1961,2.457002,Inflation
2,1962,3.117506,Inflation
3,1963,0.697674,Inflation
4,1964,-0.099305,Inflation


In [18]:
unemployment_df.head()

,year,value,indicator
0,1991,2.780,Unemployment
1,1992,2.894,Unemployment
2,1993,2.979,Unemployment
3,1994,2.955,Unemployment
4,1995,2.848,Unemployment


## Step 3: Combining the three indicators into one dataset

**What we did**
Each indicator (GDP growth, Inflation, Unemployment) was pulled separately
into its own DataFrame, but all three share the exact same column structure:
`year`, `value`, `indicator`. Because the shape matched, we could stack them
on top of each other into one long table using `pd.concat()`:

    combined_df = pd.concat([gdp_df, inflation_df, unemployment_df], ignore_index=True)

This gave us **164 rows** — exactly 64 (GDP) + 65 (Inflation) + 35
(Unemployment), confirming nothing got dropped or duplicated in the process.

**Why this shape, and not three separate tables side by side**
This is called "long format" — instead of one row per year with separate
columns for GDP/Inflation/Unemployment, every row is a single
(year, indicator, value) observation, and the `indicator` column tells you
which series it belongs to. This matters for two reasons:

1. **It's what plotting libraries want.** Tools like Plotly/Streamlit can
   take this long format and automatically draw one line per indicator,
   color-coded, just by telling it to group on the `indicator` column —
   no manual reshaping needed later.
2. **It scales.** When CBK/KNBS data gets added later (mobile money,
   credit growth, CPI), it just becomes more rows in this same table,
   not new columns bolted onto an already-wide one. The structure doesn't
   need to change as the dashboard grows.

**Why `ignore_index=True` specifically**
Without it, each original DataFrame's row numbers (0, 1, 2...) would have
stayed attached, so the combined table would have *three separate* sets of
row 0, row 1, row 2, etc. — confusing and not useful as a unique row ID.
`ignore_index=True` throws away the old indexes and assigns one fresh,
continuous index (0 to 163) across the whole combined table.

**Sanity check worth running**

    combined_df["indicator"].value_counts()

This should show 64 / 65 / 35 — matching each original DataFrame's row
count exactly, confirming the combine didn't silently lose or duplicate
anything.

In [19]:
combined_df = pd.concat([gdp_df, inflation_df, unemployment_df], ignore_index=True)
combined_df.shape   # should be 64 + 65 + 35 = 164 rows

(164, 3)

In [20]:
import os
os.makedirs("data", exist_ok=True)

In [21]:
combined_df.to_csv("data/worldbank_indicators.csv", index=False)

In [22]:
check_df = pd.read_csv("data/worldbank_indicators.csv")
print(check_df.shape)
check_df.head()

(164, 3)


,year,value,indicator
0,1961,-7.774635,GDP growth
1,1962,9.457359,GDP growth
2,1963,8.778340,GDP growth
3,1964,4.964467,GDP growth
4,1965,2.009094,GDP growth


In [23]:
import pandas as pd

raw = pd.read_csv("data/raw/274043565_Depository Corporation Survey - CSV.csv", header=None)
raw.shape

(122, 149)

In [24]:
target_row = raw[raw[1].astype(str).str.strip() == "Private sector credit*"]
target_row

,0,1,2,3,4,5,6,7,8,9,...,139,140,141,142,143,144,145,146,147,148
112,NaN,Private sector credit*,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.429287034,5.722668515,5.465610862,6.589831519,7.381716006,7.411506054,7.321271092,7.432386811,8.191749864,8.461412402


In [25]:
#  look at both header rows first
years_row = raw.iloc[0]
months_row = raw.iloc[2]

years_row.head(20)
months_row.head(20)

0     NaN
1     NaN
2     Jan
3     Feb
4     Mar
5     Apr
6     May
7     Jun
8     Jul
9     Aug
10    Sep
11    Oct
12    Nov
13    Dec
14    Jan
15    Feb
16    Mar
17    Apr
18    May
19    Jun
Name: 2, dtype: object

In [26]:
#  fix the year row so every column has a year, not just the first of each block.
years_filled = years_row.ffill()
years_filled.head(20)

0     **BANKING   SURVEY - AT CURRENT EXCHANGE RATE
1     **BANKING   SURVEY - AT CURRENT EXCHANGE RATE
2                                              2014
3                                              2014
4                                              2014
5                                              2014
6                                              2014
7                                              2014
8                                              2014
9                                              2014
10                                             2014
11                                             2014
12                                             2014
13                                             2014
14                                             2015
15                                             2015
16                                             2015
17                                             2015
18                                             2015
19          

In [27]:
# combine year + month into one label per column.
labels = [f"{int(y)}-{m}" for y, m in zip(years_filled, months_row) if pd.notna(y) and pd.notna(m)]
len(labels)

147

In [28]:
values = target_row.iloc[0, 2:]   # drop the first 2 columns, keep only the data
len(values)

147

In [29]:
credit_df = pd.DataFrame({"period": labels, "value": values.values})
credit_df = credit_df.dropna(subset=["value"]).reset_index(drop=True)
credit_df.tail(10)

,period,value
125,2025-Jun,4.429287034
126,2025-Jul,5.722668515
127,2025-Aug,5.465610862
128,2025-Sept,6.589831519
129,2025-Oct,7.381716006
130,2025-Nov,7.411506054
131,2025-Dec,7.321271092
132,2026-Jan,7.432386811
133,2026-Feb,8.191749864
134,2026-Mar,8.461412402


In [30]:
len(credit_df)
credit_df.head()

,period,value
0,2015-Jan,26.15433651
1,2015-Feb,25.35195038
2,2015-Mar,23.89613962
3,2015-Apr,23.89024028
4,2015-May,24.51604613


In [31]:
# Split the "2015-Jan" style string on the dash, take the FIRST piece (index 0),
# and convert it from text to an actual integer so it behaves like a real year
# (sortable, filterable, plottable) instead of just a string that looks like one.
credit_df["year"] = credit_df["period"].str.split("-").str[0].astype(int)

# Same split, but take the SECOND piece (index 1) instead — this gives us the
# month name on its own. We're keeping this as text (not converting to a number)
# because month names like "Jan" are more readable in tables/charts than "1".
credit_df["month"] = credit_df["period"].str.split("-").str[1]

# Stamp every row with a label saying which indicator this data represents.
# This isn't calculated from anything — we're just hardcoding it because this
# whole DataFrame only ever holds ONE indicator (private sector credit growth).
# This column is what will let us eventually combine this with other CBK
# indicators later, the same way "indicator" did for your World Bank data.
credit_df["indicator"] = "Private sector credit growth (%)"

# Reorder/select only the columns we actually want going forward.
# We're dropping the original "period" column here since we've now split it
# into "year" and "month" separately — keeping both would just be duplicate
# information sitting around.
credit_df = credit_df[["year", "month", "value", "indicator"]]

# Just a sanity check — look at the first 5 rows to confirm everything above
# actually worked the way we expected, before trusting it with anything else.
credit_df.head()

,year,month,value,indicator
0,2015,Jan,26.15433651,Private sector credit growth (%)
1,2015,Feb,25.35195038,Private sector credit growth (%)
2,2015,Mar,23.89613962,Private sector credit growth (%)
3,2015,Apr,23.89024028,Private sector credit growth (%)
4,2015,May,24.51604613,Private sector credit growth (%)


In [32]:
credit_df.tail()

,year,month,value,indicator
130,2025,Nov,7.411506054,Private sector credit growth (%)
131,2025,Dec,7.321271092,Private sector credit growth (%)
132,2026,Jan,7.432386811,Private sector credit growth (%)
133,2026,Feb,8.191749864,Private sector credit growth (%)
134,2026,Mar,8.461412402,Private sector credit growth (%)


In [33]:
len(credit_df)

135

In [34]:
# Make sure the destination folder actually exists before trying to write into it.
# You've already got a "data" folder, but "processed" is a new subfolder we
# haven't created yet — same situation as before with the missing "data" folder.
import os
os.makedirs("data/processed", exist_ok=True)

# Save it as its own file, separate from your World Bank CSV — these stay apart
# because they're different frequencies (monthly vs annual), not because
# anything went wrong.
credit_df.to_csv("data/processed/cbk_credit_growth.csv", index=False)

In [35]:
#  verify it actually saved correctly
check = pd.read_csv("data/processed/cbk_credit_growth.csv")
check.shape

(135, 4)

In [36]:
check.head()

,year,month,value,indicator
0,2015,Jan,26.154337,Private sector credit growth (%)
1,2015,Feb,25.351950,Private sector credit growth (%)
2,2015,Mar,23.896140,Private sector credit growth (%)
3,2015,Apr,23.890240,Private sector credit growth (%)
4,2015,May,24.516046,Private sector credit growth (%)


In [37]:
check.head()

,year,month,value,indicator
0,2015,Jan,26.154337,Private sector credit growth (%)
1,2015,Feb,25.351950,Private sector credit growth (%)
2,2015,Mar,23.896140,Private sector credit growth (%)
3,2015,Apr,23.890240,Private sector credit growth (%)
4,2015,May,24.516046,Private sector credit growth (%)


In [38]:
# Check the dtypes before anything
mobile_df = pd.read_excel("data/raw/Mobile Payments.xlsx")
mobile_df.shape
mobile_df.dtypes

Year                                                   int64
Month                                                 object
Active Agents                                          int64
Total Registered Mobile Money Accounts (Millions)    float64
Total Agent Cash in Cash Out (Volume Million)        float64
Total Agent Cash in Cash Out (Value KSh billions)    float64
dtype: object

In [39]:
# Pull out just the columns we need, and rename to match the lowercase
# "year"/"month" convention you already used in credit_df — small thing,
# but keeping naming consistent across datasets avoids silly bugs later
# when you try to combine or compare them.
mobile_df = mobile_df.rename(columns={"Year": "year", "Month": "month"})

# Select just the one indicator we decided on, plus year/month.
mobile_value_df = mobile_df[["year", "month", "Total Agent Cash in Cash Out (Value KSh billions)"]].copy()

# Rename that long column name to something more usable, and tag it
# with an "indicator" label the same way you did for credit growth.
mobile_value_df = mobile_value_df.rename(columns={"Total Agent Cash in Cash Out (Value KSh billions)": "value"})
mobile_value_df["indicator"] = "Mobile money transaction value (KSh billions)"

mobile_value_df.head()

,year,month,value,indicator
0,2026,May,681.45,Mobile money transaction value (KSh billions)
1,2026,April,680.99,Mobile money transaction value (KSh billions)
2,2026,March,693.37,Mobile money transaction value (KSh billions)
3,2026,February,633.35,Mobile money transaction value (KSh billions)
4,2026,January,699.64,Mobile money transaction value (KSh billions)


In [40]:
mobile_value_df.tail()

,year,month,value,indicator
226,2007,July,1.065370,Mobile money transaction value (KSh billions)
227,2007,June,0.720102,Mobile money transaction value (KSh billions)
228,2007,May,0.483709,Mobile money transaction value (KSh billions)
229,2007,April,0.220896,Mobile money transaction value (KSh billions)
230,2007,March,0.064391,Mobile money transaction value (KSh billions)


In [41]:
# Same idea as before — pull year/month plus the registered-accounts column,
# and reshape it into the same year/month/value/indicator shape as everything
# else you've built so far. Consistency here is what makes combining easy later.
mobile_accounts_df = mobile_df[["year", "month", "Total Registered Mobile Money Accounts (Millions)"]].copy()

mobile_accounts_df = mobile_accounts_df.rename(
    columns={"Total Registered Mobile Money Accounts (Millions)": "value"}
)
mobile_accounts_df["indicator"] = "Registered mobile money accounts (millions)"

mobile_accounts_df.head()

,year,month,value,indicator
0,2026,May,94.09,Registered mobile money accounts (millions)
1,2026,April,92.84,Registered mobile money accounts (millions)
2,2026,March,91.39,Registered mobile money accounts (millions)
3,2026,February,91.32,Registered mobile money accounts (millions)
4,2026,January,90.41,Registered mobile money accounts (millions)


In [42]:
mobile_accounts_df.tail()

,year,month,value,indicator
226,2007,July,0.268499,Registered mobile money accounts (millions)
227,2007,June,0.175652,Registered mobile money accounts (millions)
228,2007,May,0.107733,Registered mobile money accounts (millions)
229,2007,April,0.054944,Registered mobile money accounts (millions)
230,2007,March,0.020992,Registered mobile money accounts (millions)


In [43]:
# Stack the two mobile money indicators into one table — same idea as your
# earlier pd.concat() for GDP/inflation/unemployment.
mobile_combined_df = pd.concat([mobile_value_df, mobile_accounts_df], ignore_index=True)

# This data currently runs newest-first (2026 at the top), same issue you
# fixed on your very first World Bank pull. Sort it properly before saving.
mobile_combined_df = mobile_combined_df.sort_values(["indicator", "year", "month"]).reset_index(drop=True)

mobile_combined_df.shape   # should be roughly 231 + 231 = 462 rows

(462, 4)

In [44]:
mobile_combined_df.head()

,year,month,value,indicator
0,2007,April,0.220896,Mobile money transaction value (KSh billions)
1,2007,August,1.579910,Mobile money transaction value (KSh billions)
2,2007,December,3.770270,Mobile money transaction value (KSh billions)
3,2007,July,1.065370,Mobile money transaction value (KSh billions)
4,2007,June,0.720102,Mobile money transaction value (KSh billions)


In [45]:
# Pandas needs to know "January" = 1, "February" = 2, etc. to sort correctly —
# it has no built-in understanding that calendar months have a real order.
month_order = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}

mobile_combined_df["month_num"] = mobile_combined_df["month"].map(month_order)

# Now sort using the actual numeric month, not the text name.
mobile_combined_df = mobile_combined_df.sort_values(
    ["indicator", "year", "month_num"]
).reset_index(drop=True)

mobile_combined_df.head(15)

,year,month,value,indicator,month_num
0,2007,March,0.064391,Mobile money transaction value (KSh billions),3
1,2007,April,0.220896,Mobile money transaction value (KSh billions),4
2,2007,May,0.483709,Mobile money transaction value (KSh billions),5
3,2007,June,0.720102,Mobile money transaction value (KSh billions),6
4,2007,July,1.065370,Mobile money transaction value (KSh billions),7
5,2007,August,1.579910,Mobile money transaction value (KSh billions),8
6,2007,September,2.069690,Mobile money transaction value (KSh billions),9
7,2007,October,2.829550,Mobile money transaction value (KSh billions),10
8,2007,November,3.514950,Mobile money transaction value (KSh billions),11
9,2007,December,3.770270,Mobile money transaction value (KSh billions),12


In [46]:
mobile_combined_df["month_num"].isna().sum()

np.int64(0)

In [47]:
# Drop month_num — it was only ever a sorting tool, not real data we want
# to keep. No point storing it in the final file.
mobile_combined_df = mobile_combined_df.drop(columns=["month_num"])

# Save to its own file, separate from your World Bank and credit growth CSVs —
# same "processed" folder you already created.
mobile_combined_df.to_csv("data/processed/cbk_mobile_money.csv", index=False)

# Verify it round-trips correctly — same check you did for the other files.
check = pd.read_csv("data/processed/cbk_mobile_money.csv")
print(check.shape)
check.head()

(462, 4)


,year,month,value,indicator
0,2007,March,0.064391,Mobile money transaction value (KSh billions)
1,2007,April,0.220896,Mobile money transaction value (KSh billions)
2,2007,May,0.483709,Mobile money transaction value (KSh billions)
3,2007,June,0.720102,Mobile money transaction value (KSh billions)
4,2007,July,1.065370,Mobile money transaction value (KSh billions)
